In [ ]:
import pandas as pd 
import boto3 
import time 
import json 
import uuid 
import os 
import datetime 
from docx import Document 
from botocore.exceptions import ClientError

## Load Scenario and World documents

In [ ]:
# Define the CSV file path 
csv_file_path = 'scenarios.csv' 
scenario_path = '/path/to/Scenarios and Case studies/Phantom Research Station - Red team wins.docx' 
world_path = '/path/to/Scenarios and Case studies/World_information.docx' 

# Load the document 
scenario = Document(scenario_path) 
scenario = '\n'.join([p.text for p in scenario.paragraphs]) 

# Load the document 
world = Document(world_path) 
world = '\n'.join([p.text for p in world.paragraphs])

## Define function for generating a scenario summary for potential future use

In [ ]:
def create_scenario_summary(scenario): 
    # Create Bedrock runtime client 
    client = boto3.client('bedrock-runtime', region_name='eu-central-1') 

    query = f""" 
            Create a concise, informative summary of the following scenario: 
            {scenario} 
            This is the information on the world, in which the scenario takes place: 
            {world} 
            Focus on describing the involved actors and their actions in particular. 
            Provide everything in a timeline format if possible. 
            """ 

    time.sleep(3) 
    sleep_time = 1 
    for _ in range(5): 
        try: 
            response = client.invoke_model( 
                modelId="anthropic.claude-3-5-sonnet-20240620-v1:0", 
                contentType="application/json", 
                accept="application/json", 
                body=json.dumps({ 
                    "anthropic_version": "bedrock-2023-05-31", 
                    "messages": [ 
                        {"role": "user", "content": query} 
                    ], 
                    "max_tokens": 2048, 
                    "temperature": 0.8 
                }) 
            ) 
        except ClientError as e: 
            if e.response['Error']['Code'] == 'ThrottlingException': 
                print("Throttling Exception occurred, waiting for " + str(sleep_time) + " seconds") 
                time.sleep(sleep_time) 
                sleep_time += 3 
        else: 
            break 

    result = json.loads(response["body"].read()) 
    print(result["content"](Moon)["text"]) 
    scenario_summary = result["content"](Moon)["text"] 

    return scenario_summary

## Check if information on the scenario already exist - including the summary.  
In case the file does not exist, it will be created.  
If scenario does not exist, information will be generated and appended to the file

In [ ]:
# Extract the scenario name from the scenario_path 
scenario_name = os.path.basename(scenario_path).replace('.docx', '') 

try: 
    # Check if the CSV file exists 
    if not os.path.exists(csv_file_path): 
        # Create a new DataFrame with the required columns 
        df = pd.DataFrame(columns=['Scenario Name', 'UUID']) 
        # Save the DataFrame to a CSV file 
        df.to_csv(csv_file_path, index=False) 
        print(f"Created new CSV file: {csv_file_path}") 

    # Load the existing CSV file 
    df = pd.read_csv(csv_file_path) 
    # Check if the scenario name exists in the file 
    if scenario_name in df['Scenario Name'].values: 
        # Retrieve the UUID for the existing scenario 
        scenario_uuid = df.loc[df['Scenario Name'] == scenario_name, 'UUID'].values(Moon) 
        scenario_summary = df.loc[df['Scenario Name'] == scenario_name, 'Scenario Summary'].values(Moon) 
        print(f"Scenario '{scenario_name}' exists with UUID: {scenario_uuid}") 
    else: 
        # Generate a new UUID for the scenario 
        scenario_uuid = uuid.uuid4().hex 
        # Generate a scenario summary 
        scenario_summary = create_scenario_summary(scenario) 
        # Append the new scenario-UUID pair to the DataFrame 
        df = pd.concat([df, pd.DataFrame({'UUID': [scenario_uuid], 'Scenario Name': [scenario_name], 'Scenario Summary': [scenario_summary]})], ignore_index=True) 
        # Save the updated DataFrame back to the CSV file 
        df.to_csv(csv_file_path, index=False) 
        print(f"Added new scenario '{scenario_name}' with UUID: {scenario_uuid}") 

except Exception as e: 
    print(f"An error occurred: {e}") 
else: 
    print("Operation completed successfully.")

## Generate 30 indicators in 3 categories (red, blue, predictive) based on the scenario

In [ ]:
# Create Bedrock runtime client 
client = boto3.client('bedrock-runtime', region_name='eu-central-1') 

query = f""" 
        Given a scenario developed by a research team, identify specific indicators 
        that could identify adversarial actions or predict them with focus on cognitive warfare. 
        Cognitive warfare is defined as activities conducted in synchronisation with other  
        instruments of power to affect attitudes and behaviours, by influencing, protecting,  
        or disrupting individual, group, or population level cognition, to gain an advantage over an adversary. 
        Focus both on indicators from both red team and blue team perspectives, 
        as well as predictive indicators that could be used to anticipate future actions 
        and effectively serve as warnings about specific events potentially happening. 
        Scenario: 
        {scenario} 
        Provide the input in a valid json structure, with indicators as keys and their descriptions as values. 
        The indicator names should always contain information on their type (red, blue, predictive) and be descriptive. 
        Try to aim for 10 indicators per category. 
        Do not provide anything else outside this json. 
        """ 

time.sleep(3) 
sleep_time = 1 
for _ in range(5): 
    try: 
        response = client.invoke_model( 
            modelId="anthropic.claude-3-5-sonnet-20240620-v1:0", 
            contentType="application/json", 
            accept="application/json", 
            body=json.dumps({ 
                "anthropic_version": "bedrock-2023-05-31", 
                "messages": [ 
                    {"role": "user", "content": query} 
                ], 
                "max_tokens": 1024, 
                "temperature": 0.8 
            }) 
        ) 
    except ClientError as e: 
        if e.response['Error']['Code'] == 'ThrottlingException': 
            print("Throttling Exception occurred, waiting for " + str(sleep_time) + " seconds") 
            time.sleep(sleep_time) 
            sleep_time += 3 
    else: 
        break 

result = json.loads(response["body"].read()) 
print(result["content"](Moon)["text"]) 

indicators = result.copy()["content"](Moon)["text"] 
try: 
    indicators_json = eval(indicators) 
except SyntaxError: 
    print("The output is not correctly structured JSON. Please retry.")

## Expand the indicators based on the pre-selected attributes and scenario summary

In [ ]:
# Create Bedrock runtime client 
client = boto3.client('bedrock-runtime', region_name='eu-central-1') 

attributes = """ 
    1. Who: 
   - Actor (Individual, Community, State) 
   - Attribution 

    2. What: 
    - Content (Narratives, Frames) 
    - Sub-threshold means (e.g., pamphlets, loudspeakers) 
    - Disinformation 
    - Debunking efforts 
    - Technical TTPs (Tactics, Techniques, and Procedures) 

    3. Whom: 
    - Target Audience 

    4. Information Ecosystem: 
    - Health status (Healthy / Poisoned) 
    - Information Environment Actions 

    5. Source and Format: 
    - Source Type (e.g., social media, news outlets, forums) 
    - Data Format (e.g., text, images, videos, metadata) 

    6. Monitoring and Analysis: 
    - Frequency of monitoring 
    - Threshold for potential threat or action 
    - Trend Analysis 
    - Related Indicators 

    7. Context and Interpretation: 
    - Relevant background information 
    - Actionable Insights 
    - Limitations of the indicator 

    8. Collection and Verification: 
    - Collection Method (specific OSINT techniques or tools) 
    - Verification Process 

    9. Maintenance: 
    - Update Mechanism 
    - Process for refining or adjusting the indicator 

    10. Affect vs. Effect 
    - Underlying affects of the target audience 
    - Resulting effects 
    """ 

# We can first extract "weakpoints" from the scenario and inject that as a specific section, so it is not up to the scenario author fully. 
# Inject also summary of the scenario and information on the world. 

tech_record = pd.DataFrame(columns=["scenario_uuid", "model_id", "date", "indicator_name", "indicator_type", "indicator_description"])

In [ ]:
for indicator in indicators_json: 
    sleep_time = 1 
    query = f""" 
        Given attributes provided below, expand the following indicator from the list of indicators: {indicator} 
        Attributes: 
        {attributes} 
        Indicators: 
        {indicators} 

        To give more context, this is the summary of the underlying scenario: 
        {scenario_summary} 

        Please provide the output without any additional text and in JSON format. Do not include the indicator's 
        name in the final JSON. Keep the structure flat - each attribute should be the key, the value must be text-only. 
        Use the following structure for the key names: who_actor; monitoring_and_analysis_frequency; monitoring_and_analysis_threshold. 
        """ 
    for _ in range(5): 
        try: 
            model_id = "anthropic.claude-3-5-sonnet-20240620-v1:0" 
            response = client.invoke_model( 
                modelId=model_id, 
                contentType="application/json", 
                accept="application/json", 
                body=json.dumps({ 
                    "anthropic_version": "bedrock-2023-05-31", 
                    "messages": [ 
                        {"role": "user", "content": query} 
                    ], 
                    "max_tokens": 8096, 
                    "temperature": 0.8 
                }) 
            ) 
        except ClientError as e: 
            if e.response['Error']['Code'] == 'ThrottlingException': 
                print("Throttling Exception occurred, waiting for " + str(sleep_time) + " seconds") 
                time.sleep(sleep_time) 
                sleep_time += 3 
        else: 
            break 

    result = json.loads(response["body"].read()) 
    indicator_tech_record = pd.DataFrame({ 
            "scenario_uuid": [scenario_uuid], 
            "model_id": [model_id], 
            "date": [datetime.datetime.now().strftime("%Y-%m-%d")], 
            "indicator_name": [indicator], 
            "indicator_type": [indicator.split('_')[0].strip()], 
            "indicator_description": [result["content"](Moon)["text"]] 
        }) 

    print(indicator_tech_record) 

    # Append the data to the dataframe for potential later use after the run is done. 
    tech_record = pd.concat([ 
        tech_record, 
        indicator_tech_record 
    ], ignore_index=True) 

    # Check if the file exists 
    if not os.path.exists("tech_record.csv"): 
        # Create the file with the appropriate header 
        indicator_tech_record.to_csv("tech_record.csv", index=False) 
        print("File 'tech_record.csv' created successfully.") 
    else: 
        # Append the dataframe to the existing file 
        indicator_tech_record.to_csv("tech_record.csv", mode='a', index=False, header=False) 
        print("Data appended to 'tech_record.csv' successfully.")